# Lektion 07 - Planungsmuster

Dieses Notebook demonstriert das **Planungsmuster** für KI-Agenten unter Verwendung des Microsoft Agent Frameworks.
Sie lernen, wie man komplexe Reiseanfragen in strukturierte Teilaufgaben aufteilt, diese Spezialagenten zuweist
und den resultierenden Plan ausführt — alles unter Verwendung von strukturiertem Output, der durch Pydantic-Modelle unterstützt wird.


## Einrichtung


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Aufgabenzerlegung

Die Aufgabenzerlegung ist der Kern des Planungsmusters. Anstatt einen einzelnen Agenten zu bitten, eine komplexe Anfrage von Anfang bis Ende zu bearbeiten, zerlegen wir das Problem in kleinere, klar definierte **Teilaufgaben**. Jede Teilaufgabe wird einem spezialisierten Agenten zugewiesen (z. B. Flüge, Hotels, Aktivitäten) mit klaren Prioritäten und Abhängigkeitsreihenfolge.

Dieser Ansatz bietet mehrere Vorteile:
- **Klarheit**: Jede Teilaufgabe hat eine einzige Verantwortlichkeit.
- **Parallelität**: Unabhängige Teilaufgaben können gleichzeitig ausgeführt werden.
- **Zuverlässigkeit**: Fehler sind auf einzelne Teilaufgaben isoliert.
- **Budgetüberwachung**: Kosten werden pro Teilaufgabe geschätzt und zusammengefasst.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Erstellen eines Planungsagenten mit strukturiertem Output

Der Planungsagent fungiert als **Empfangskoordinator**. Ausgehend von einer groben Reiseanfrage erstellt er einen strukturierten `TravelPlan` — er zerlegt die Anfrage in Teilaufgaben, setzt Prioritäten und identifiziert Abhängigkeiten, sodass ein Concierge oder eine Ausführungsebene die Arbeit übernehmen kann.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Ausführung eines Plans mit Spezialwerkzeugen

Sobald der Empfangsmitarbeiter einen strukturierten Plan erstellt hat, führt der **Concierge-Agent** diesen aus.  
Jedes Spezialwerkzeug bearbeitet eine Kategorie von Teilaufgaben (Flüge, Hotels, Aktivitäten). Der Concierge durchläuft die Teilaufgaben des Plans in Abhängigkeitsreihenfolge und weist jede einzelne dem entsprechenden Werkzeug zu.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Zusammenfassung

In dieser Lektion haben Sie das **Planungs-Designmuster** für KI-Agenten kennengelernt:

1. **Aufgabenzerlegung** — Ein Front-Desk-Planungsagent zerlegt eine komplexe Reiseanfrage in strukturierte Teilaufgaben mithilfe von Pydantic-Modellen und weist jede einem Spezialistenagenten mit Prioritäten und Abhängigkeiten zu.
2. **Strukturierte Ausgabe** — Durch die Übergabe eines `response_format` liefert der Agent ein validiertes `TravelPlan`-Objekt anstelle von Freitext, was die nachgelagerte Verarbeitung zuverlässig macht.
3. **Plan-Ausführung** — Ein Concierge-Agent arbeitet die Teilaufgaben unter Verwendung von Spezialwerkzeugen (`book_flight`, `reserve_hotel`, `book_activity`) ab, um den Plan auszuführen und Ergebnisse zu melden.

Dieses Muster trennt *was zu tun ist* (Planung) von *wie es zu tun ist* (Ausführung), wodurch Agenten modularer, testbarer und leichter erweiterbar werden.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, sollten Sie beachten, dass automatische Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Für wichtige Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Nutzung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
